# Workstream 1 — Final Data Validation

Use this notebook **after rerunning the main Workstream 1 notebook with fresh HCD data**.

This validation notebook:

- reads only the processed files that feed Power BI;
- uses exact jurisdiction filters (never `str.contains("San Diego")`);
- keeps RHNA, applications, entitlements, permits, and completions separate;
- validates housing-type totals against the main Power BI fact table;
- validates RHNA arithmetic and DOF housing-stock reconciliation;
- produces one clean 45-row validation table for manual review.

**RHNA methodology:** RHNA progress is based on building permits, not completions.


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)


In [2]:
def find_workstream_root() -> Path:
    cwd = Path.cwd().resolve()

    for candidate in [cwd, *cwd.parents]:
        if (
            (candidate / "notebooks").exists()
            and (candidate / "data" / "processed").exists()
        ):
            return candidate

        van_folder = candidate / "van's work"
        if (
            (van_folder / "notebooks").exists()
            and (van_folder / "data" / "processed").exists()
        ):
            return van_folder

    raise FileNotFoundError(
        "Could not find the Workstream 1 folder containing notebooks/ "
        "and data/processed/."
    )

ROOT = find_workstream_root()
PROCESSED = ROOT / "data" / "processed"

print("Workstream root:", ROOT)
print("Processed folder:", PROCESSED)


Workstream root: /Users/laurenvo/Documents/Github/chpd-dashboard-data-validation/van's work
Processed folder: /Users/laurenvo/Documents/Github/chpd-dashboard-data-validation/van's work/data/processed


In [3]:
# Load the exact Power BI / processed outputs.

rhna = pd.read_csv(
    PROCESSED / "rhna_housing_production_2025_by_jurisdiction.csv"
)

powerbi = pd.read_csv(
    PROCESSED / "powerbi_rhna_production_2018_2025_long.csv"
)

housing_type = pd.read_csv(
    PROCESSED / "powerbi_production_by_housing_type_2018_2025.csv"
)

benchmarks = pd.read_csv(
    PROCESSED / "powerbi_housing_stock_benchmarks_2000_2010_2021_2024.csv"
)

dof = pd.read_csv(
    PROCESSED / "powerbi_dof_annual_housing_stock_2020_2025.csv"
)

print("rhna:", rhna.shape)
print("powerbi:", powerbi.shape)
print("housing_type:", housing_type.shape)
print("benchmarks:", benchmarks.shape)
print("dof:", dof.shape)


rhna: (19, 61)
powerbi: (3760, 12)
housing_type: (2560, 12)
benchmarks: (80, 14)
dof: (1200, 12)


In [4]:
# ============================================================
# STRUCTURAL / GRAIN CHECKS
# ============================================================

assert len(rhna) == 19, "RHNA summary should contain exactly 19 jurisdictions."
assert rhna["jur_clean"].nunique() == 19

assert not powerbi.duplicated(
    ["jurisdiction", "year", "development_stage", "income_category"]
).any(), "Duplicate keys in main Power BI fact table."

assert not housing_type.duplicated(
    ["jurisdiction", "year", "development_stage", "housing_type"]
).any(), "Duplicate keys in housing-type Power BI table."

assert (
    benchmarks.groupby("benchmark_year").size().eq(20).all()
), "Each benchmark year should have 19 jurisdictions + 1 countywide row."

assert not benchmarks.duplicated(
    ["jurisdiction", "benchmark_year"]
).any()

assert not dof.duplicated(
    ["jurisdiction", "year", "metric"]
).any()

assert (
    dof.groupby("year").size().eq(200).all()
), "Each DOF year should have 20 geographies x 10 metrics = 200 rows."

print("Structural checks passed.")


Structural checks passed.


In [5]:
# ============================================================
# RHNA ARITHMETIC CHECKS — ALL 19 JURISDICTIONS
# ============================================================

tiers = ["very_low", "low", "moderate", "above_moderate"]

assert np.allclose(
    rhna[[f"rhna_target_{t}" for t in tiers]].sum(axis=1),
    rhna["rhna_target_total"],
    equal_nan=True,
)

assert np.allclose(
    rhna[[f"rhna_reported_{t}" for t in tiers]].sum(axis=1),
    rhna["rhna_units_reported_total"],
    equal_nan=True,
)

# Remaining total follows the notebook's stated rule:
# sum of tier-specific remaining obligations, with each tier clipped at zero.
assert np.allclose(
    rhna[[f"rhna_remaining_{t}" for t in tiers]].sum(axis=1),
    rhna["rhna_remaining_total"],
    equal_nan=True,
)

assert np.allclose(
    rhna["rhna_units_reported_total"] / rhna["rhna_target_total"],
    rhna["rhna_pct_achieved"],
    equal_nan=True,
)

for tier in tiers:
    assert np.allclose(
        rhna[f"rhna_reported_{tier}"] / rhna[f"rhna_target_{tier}"],
        rhna[f"rhna_pct_achieved_{tier}"],
        equal_nan=True,
    )

print("RHNA arithmetic checks passed for all 19 jurisdictions.")


RHNA arithmetic checks passed for all 19 jurisdictions.


In [6]:
# ============================================================
# HOUSING-TYPE TOTALS MUST RECONCILE TO POWER BI STAGE TOTALS
# ============================================================

for stage in ["Building Permit", "Completion"]:
    typed = (
        housing_type[
            (housing_type["geographic_level"] == "Jurisdiction")
            & (housing_type["development_stage"] == stage)
        ]
        .groupby(["jurisdiction", "year"])["value"]
        .sum(min_count=1)
        .sort_index()
    )

    headline = (
        powerbi[
            (powerbi["geographic_level"] == "Jurisdiction")
            & (powerbi["development_stage"] == stage)
            & (powerbi["income_category"] == "all")
        ]
        .set_index(["jurisdiction", "year"])["value"]
        .sort_index()
    )

    common = typed.index.intersection(headline.index)

    assert np.allclose(
        typed.loc[common],
        headline.loc[common],
        equal_nan=True,
    ), f"{stage} housing-type totals do not reconcile."

print("Housing-type reconciliation passed for permits and completions.")


Housing-type reconciliation passed for permits and completions.


In [7]:
# ============================================================
# DOF RECONCILIATION — ALL YEARS / GEOGRAPHIES
# ============================================================

dof_wide = dof.pivot_table(
    index=["jurisdiction", "year"],
    columns="metric",
    values="value",
    aggfunc="first",
)

assert np.allclose(
    dof_wide["total_housing_units"],
    dof_wide["occupied_housing_units"] + dof_wide["vacant_housing_units"],
    equal_nan=True,
)

structure_sum = (
    dof_wide["single_family_detached"]
    + dof_wide["single_family_attached"]
    + dof_wide["two_to_four_units"]
    + dof_wide["five_plus_units"]
    + dof_wide["mobile_homes"]
)

assert np.allclose(
    dof_wide["total_housing_units"],
    structure_sum,
    equal_nan=True,
)

print("DOF occupancy and structure reconciliation passed.")


DOF occupancy and structure reconciliation passed.


In [8]:
# ============================================================
# HELPERS FOR THE FINAL 45-ROW MANUAL VALIDATION TABLE
# ============================================================

def one_row(df, mask, label):
    result = df.loc[mask]
    if len(result) != 1:
        raise AssertionError(
            f"{label}: expected exactly 1 row, found {len(result)}."
        )
    return result.iloc[0]


sd = one_row(
    rhna,
    rhna["jur_clean"].eq("san diego"),
    "San Diego RHNA summary",
)


def stage_value(year, stage):
    row = one_row(
        powerbi,
        powerbi["jurisdiction"].eq("San Diego")
        & powerbi["year"].eq(year)
        & powerbi["development_stage"].eq(stage)
        & powerbi["income_category"].eq("all")
        & powerbi["geographic_level"].eq("Jurisdiction"),
        f"San Diego {year} {stage}",
    )
    return row["value"]


def type_value(stage, housing_type_label, year=2025):
    row = one_row(
        housing_type,
        housing_type["jurisdiction"].eq("San Diego")
        & housing_type["year"].eq(year)
        & housing_type["development_stage"].eq(stage)
        & housing_type["housing_type"].eq(housing_type_label)
        & housing_type["geographic_level"].eq("Jurisdiction"),
        f"San Diego {year} {stage} / {housing_type_label}",
    )
    return row["value"]


def benchmark_value(year):
    row = one_row(
        benchmarks,
        benchmarks["jurisdiction"].eq("San Diego")
        & benchmarks["benchmark_year"].eq(year),
        f"San Diego benchmark {year}",
    )
    return row["housing_units_total"]


def dof_value(year, metric):
    row = one_row(
        dof,
        dof["jurisdiction"].eq("San Diego")
        & dof["year"].eq(year)
        & dof["metric"].eq(metric),
        f"San Diego DOF {year} / {metric}",
    )
    return row["value"]


In [9]:
# ============================================================
# FINAL 45-ROW VALIDATION TABLE
# Same metric order as the Workstream 1 manual validation sheet.
# ============================================================

rows = []

def add(metric, period, value, value_type="count"):
    rows.append(
        {
            "Metric": metric,
            "Jurisdiction": "San Diego",
            "Year / Period": period,
            "Notebook Value": value,
            "_value_type": value_type,
        }
    )

# RHNA allocation
add("RHNA Allocation - Total", "6th Cycle snapshot", sd["rhna_target_total"])
add("RHNA Allocation - Very Low Income", "6th Cycle snapshot", sd["rhna_target_very_low"])
add("RHNA Allocation - Low Income", "6th Cycle snapshot", sd["rhna_target_low"])
add("RHNA Allocation - Moderate Income", "6th Cycle snapshot", sd["rhna_target_moderate"])
add("RHNA Allocation - Above Moderate Income", "6th Cycle snapshot", sd["rhna_target_above_moderate"])

# RHNA qualifying units
add("RHNA Qualifying Units - Total", "6th Cycle snapshot", sd["rhna_units_reported_total"])
add("RHNA Qualifying Units - Very Low Income", "6th Cycle snapshot", sd["rhna_reported_very_low"])
add("RHNA Qualifying Units - Low Income", "6th Cycle snapshot", sd["rhna_reported_low"])
add("RHNA Qualifying Units - Moderate Income", "6th Cycle snapshot", sd["rhna_reported_moderate"])
add("RHNA Qualifying Units - Above Moderate Income", "6th Cycle snapshot", sd["rhna_reported_above_moderate"])

# Remaining and percentages
add("RHNA Remaining Units - Total", "6th Cycle snapshot", sd["rhna_remaining_total"])
add("RHNA Percent Complete - Total", "6th Cycle snapshot", sd["rhna_pct_achieved"], "share")
add("RHNA Percent Complete - Very Low Income", "6th Cycle snapshot", sd["rhna_pct_achieved_very_low"], "share")
add("RHNA Percent Complete - Low Income", "6th Cycle snapshot", sd["rhna_pct_achieved_low"], "share")
add("RHNA Percent Complete - Moderate Income", "6th Cycle snapshot", sd["rhna_pct_achieved_moderate"], "share")
add("RHNA Percent Complete - Above Moderate Income", "6th Cycle snapshot", sd["rhna_pct_achieved_above_moderate"], "share")

# Sample historical stage validation
add("Applications Submitted", "2020", stage_value(2020, "Application"))
add("Entitlements / Planning Approvals", "2019", stage_value(2019, "Entitlement"))
add("Building Permits Issued", "2022", stage_value(2022, "Building Permit"))
add("Completed Units", "2024", stage_value(2024, "Completion"))

# Permit rate — target year
permits_2025 = stage_value(2025, "Building Permit")
permits_per_1000 = permits_2025 / sd["population_total"] * 1000
add("Permits per 1,000 Residents", "2025", permits_per_1000, "rate")

# Production by housing type — permits
types = [
    ("Single-family detached", "SFD"),
    ("Single-family attached", "SFA"),
    ("2-4 unit building (combined)", "2-4"),
    ("5+ unit building", "5+"),
    ("Accessory dwelling unit", "ADU"),
    ("Mobile home / manufactured home", "Mobile / Manufactured"),
]

for source_label, display_label in types:
    add(
        f"Production - {display_label} Permitted",
        "2025",
        type_value("Building Permit", source_label),
    )

# Production by housing type — completions
for source_label, display_label in types:
    add(
        f"Production - {display_label} Completed",
        "2025",
        type_value("Completion", source_label),
    )

# Current housing stock
add("Total Housing Stock", "2025", sd["housing_units_total"])
add("Occupied Housing Units", "2025", sd["occupied_units"])
add("Vacant Housing Units", "2025", sd["vacant_units"])
add("Single-Family Housing Stock", "2025", sd["single_family_units"])
add("Multifamily Housing Stock", "2025", sd["multifamily_units"])
add("Mobile Home Housing Stock", "2025", sd["mobile_home_units"])

housing_per_1000 = sd["housing_units_total"] / sd["population_total"] * 1000
add("Housing Units per 1,000 Residents", "2025", housing_per_1000, "rate")

# Benchmark housing stock
for year in [2000, 2010, 2021, 2024]:
    add(
        f"Housing Stock Benchmark - {year}",
        str(year),
        benchmark_value(year),
    )

# Annual DOF stock sample
add(
    "Annual DOF Housing Stock",
    "2025",
    dof_value(2025, "total_housing_units"),
)

validation_table = pd.DataFrame(rows)

assert len(validation_table) == 45, (
    f"Expected 45 validation metrics, got {len(validation_table)}."
)

def display_value(row):
    value = row["Notebook Value"]

    if pd.isna(value):
        return ""

    if row["_value_type"] == "share":
        return f"{float(value):.2%}"

    if row["_value_type"] == "rate":
        return f"{float(value):.3f}"

    return f"{float(value):,.0f}"

validation_table["Display Value"] = validation_table.apply(
    display_value,
    axis=1,
)

validation_table = validation_table[
    [
        "Metric",
        "Jurisdiction",
        "Year / Period",
        "Notebook Value",
        "Display Value",
    ]
]

display(validation_table)

print("Validation rows:", len(validation_table))


,Metric,Jurisdiction,Year / Period,Notebook Value,Display Value
0,RHNA Allocation - Total,San Diego,6th Cycle snapshot,108036.000000,"108,036"
1,RHNA Allocation - Very Low Income,San Diego,6th Cycle snapshot,27549.000000,"27,549"
2,RHNA Allocation - Low Income,San Diego,6th Cycle snapshot,17331.000000,"17,331"
3,RHNA Allocation - Moderate Income,San Diego,6th Cycle snapshot,19319.000000,"19,319"
4,RHNA Allocation - Above Moderate Income,San Diego,6th Cycle snapshot,43837.000000,"43,837"
5,RHNA Qualifying Units - Total,San Diego,6th Cycle snapshot,36533.000000,"36,533"
6,RHNA Qualifying Units - Very Low Income,San Diego,6th Cycle snapshot,2513.000000,"2,513"
7,RHNA Qualifying Units - Low Income,San Diego,6th Cycle snapshot,2958.000000,"2,958"
8,RHNA Qualifying Units - Moderate Income,San Diego,6th Cycle snapshot,1360.000000,"1,360"
9,RHNA Qualifying Units - Above Moderate Income,San Diego,6th Cycle snapshot,29702.000000,"29,702"


Validation rows: 45


In [10]:
# Final checks that are easy to communicate in the dashboard meeting.

checks = pd.DataFrame(
    [
        ["19 jurisdiction summary rows", len(rhna) == 19],
        ["RHNA arithmetic passed", True],
        ["Housing-type totals reconcile to permits/completions", True],
        ["DOF stock totals reconcile", True],
        ["45 manual validation metrics generated", len(validation_table) == 45],
    ],
    columns=["Check", "Passed"],
)

display(checks)

assert checks["Passed"].all()
print("FINAL WORKSTREAM 1 VALIDATION CHECKS PASSED")


,Check,Passed
0,19 jurisdiction summary rows,True
1,RHNA arithmetic passed,True
2,Housing-type totals reconcile to permits/completions,True
3,DOF stock totals reconcile,True
4,45 manual validation metrics generated,True


FINAL WORKSTREAM 1 VALIDATION CHECKS PASSED


In [11]:
# Export a copy-ready CSV for the manual validation sheet.

out_path = PROCESSED / "workstream1_final_validation_table.csv"
validation_table.to_csv(out_path, index=False)

print("Saved:", out_path)


Saved: /Users/laurenvo/Documents/Github/chpd-dashboard-data-validation/van's work/data/processed/workstream1_final_validation_table.csv
